# Mock worker generator

Makes a test list of field workers for the **Worker Layer** and the partition pipeline, with the columns of the real
worker file: `staff_id, node, latitude, longitude, partition`. Change the settings below and run the whole notebook.

* **partition**: the worker's team. A worker only staffs clusters of their own partition, so every partition of the
  cabinet file gets its own workers.
* **node**: the worker's home exchange, an area of nearby cabinets within the partition.
* **latitude, longitude**: the worker's **base**, where the day starts. All workers of a node share one base, like
  in the real file.
* **staff_id**: the node code and a number, for example `N03_002`.

Busier partitions and nodes get more workers. The data is made up and only meant for testing.

## Settings
Everything you can change is in the next cell. The same settings and `SEED` always give the same file;
`Dataset/mock_workers.csv` was made with the values below.

In [ ]:
CABINET_FILE = "Dataset/mock_cabinets.csv"   # cabinets the workers serve, relative to the project root
N_WORKERS = 55                               # how many workers to make, shared out by demand
NODES_PER_PARTITION = 3                      # nodes per partition if the cabinet IDs have no node code (CAB0001)
SEED = 67
OUTPUT_FILE = "Dataset/mock_workers.csv"     # relative to the project root

In [ ]:
import math
import os
from pathlib import Path

# One thread for k-means: avoids a scikit-learn memory-leak warning on Windows/Anaconda (MKL).
os.environ.setdefault("OMP_NUM_THREADS", "1")

import folium
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# The notebook may be run from the project root or from tools/
ROOT = Path.cwd().parent if Path.cwd().name == "tools" else Path.cwd()
pd.set_option("display.width", 140)

## Load the cabinets
The cabinet ID, its position, its demand (tickets per day) and its partition are used; upper and lower case don't
matter. A file without a partition column is one partition, `ALL`. Cabinets with zero demand are skipped, as in the
clustering notebooks.

In [ ]:
CABINET_COLUMNS = ["CabinetId", "cabinetlat", "cabinetlong", "demand"]


def read_cabinets(path):
    # Read the cabinet file and give the columns we need their usual names, ignoring upper/lower case.
    df = pd.read_csv(path)
    by_lower = {str(c).strip().lower(): c for c in df.columns}
    missing = [c for c in CABINET_COLUMNS if c.lower() not in by_lower]
    if missing:
        raise ValueError(f"{path.name} has no column {', '.join(missing)}. "
                         "Needed: cabinetid, cabinetlat, cabinetlong, demand.")
    out = df.rename(columns={by_lower[c.lower()]: c for c in CABINET_COLUMNS})[CABINET_COLUMNS].copy()
    out["CabinetId"] = out["CabinetId"].astype(str).str.strip()
    out["partition"] = df[by_lower["partition"]].astype(str).str.strip() if "partition" in by_lower else "ALL"
    return out


cabinets = read_cabinets(ROOT / CABINET_FILE)
cabinets = cabinets[cabinets["demand"] > 0].reset_index(drop=True)
print(f"{len(cabinets):,} cabinets with demand, total {cabinets['demand'].sum():.1f} tickets/day")
print(cabinets.groupby("partition")["demand"].agg(["size", "sum"]).round(1).to_string())

## Nodes
A cabinet's node is the part of its ID before the first `_` (`IP_C012` → `IP`). If the IDs have no `_` (like the
mock file's `CAB0001`), the cabinets of each partition are grouped into `NODES_PER_PARTITION` areas of nearby
cabinets (k-means) and named `N01`, `N02`, ...

A node's **base** is the average position of its cabinets, weighted by demand, so it sits where most of the work is.
(In the real file the base is the real office.)

In [ ]:
def node_codes(ids):
    # The part of each ID before the first "_"; None if any ID has no "_".
    if not ids.str.contains("_").all():
        return None
    return ids.str.split("_").str[0]


codes = node_codes(cabinets["CabinetId"])
if codes is None:
    codes = pd.Series("", index=cabinets.index)
    first = 1
    for name, part in cabinets.groupby("partition"):
        k = min(NODES_PER_PARTITION, len(part))
        groups = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(
            part[["cabinetlat", "cabinetlong"]], sample_weight=part["demand"]).labels_
        codes[part.index] = [f"N{first + g:02d}" for g in groups]
        first += k
    print(f"The cabinet IDs have no node code: made {first - 1} nodes from nearby cabinets")
elif cabinets["partition"].nunique() > 1:   # the same node code in two partitions is two nodes
    codes = cabinets["partition"] + "_" + codes
cabinets["node"] = codes.to_numpy()

weighted = cabinets.assign(wlat=cabinets["cabinetlat"] * cabinets["demand"],
                           wlong=cabinets["cabinetlong"] * cabinets["demand"])
nodes = weighted.groupby("node").agg(partition=("partition", "first"), cabinets=("CabinetId", "size"),
                                     demand=("demand", "sum"), wlat=("wlat", "sum"), wlong=("wlong", "sum"))
nodes["lat"] = nodes["wlat"] / nodes["demand"]
nodes["long"] = nodes["wlong"] / nodes["demand"]
nodes = nodes.drop(columns=["wlat", "wlong"])
print(f"{len(nodes)} nodes in {nodes['partition'].nunique()} partition(s)")

## Workers per node
The `N_WORKERS` workers are shared out in proportion to demand: first between the partitions, then between the
nodes of each partition. Each one first gets the whole part of its share; the few workers left go to the biggest
leftover fractions, so the counts always add up exactly.

In [ ]:
def share_out(weights, n):
    # Split n whole workers in proportion to weights (largest remainder method).
    exact = weights / weights.sum() * n
    count = np.floor(exact).astype(int)
    order = np.argsort(-(exact - count), kind="stable")
    count[order[:n - count.sum()]] += 1
    return count


per_partition = nodes.groupby("partition")["demand"].sum()
nodes["workers"] = 0
for name, n in zip(per_partition.index, share_out(per_partition.to_numpy(), N_WORKERS)):
    here = nodes["partition"] == name
    nodes.loc[here, "workers"] = share_out(nodes.loc[here, "demand"].to_numpy(), int(n))
nodes.sort_values(["partition", "demand"], ascending=[True, False]).round(3)

## Make the workers
Every worker gets their node's base as latitude and longitude, and their node's partition.

In [ ]:
def make_workers(nodes):
    # One row per worker: staff_id = node code + number, latitude/longitude = the node's base (shared).
    rows = []
    for p in nodes.itertuples():
        for k in range(1, p.workers + 1):
            rows.append({"staff_id": f"{p.Index}_{k:03d}", "node": p.Index, "latitude": round(p.lat, 6),
                         "longitude": round(p.long, 6), "partition": p.partition})
    return pd.DataFrame(rows, columns=["staff_id", "node", "latitude", "longitude", "partition"])


workers = make_workers(nodes)
workers.head()

## Checks
Exactly `N_WORKERS` rows, the five columns in order, a unique `staff_id` for everyone, every worker at their node's
base, and workers in every partition.

In [ ]:
assert len(workers) == N_WORKERS, "wrong number of workers"
assert list(workers.columns) == ["staff_id", "node", "latitude", "longitude", "partition"], "wrong columns"
assert workers["staff_id"].is_unique, "staff_id must be unique"
bases = workers.groupby("node")[["latitude", "longitude"]].nunique()
assert (bases == 1).all().all(), "workers of one node must share one base"
missing = set(cabinets["partition"]) - set(workers["partition"])
assert not missing, f"no workers in partition(s) {', '.join(sorted(missing))}: raise N_WORKERS"
print(f"Check passed: {len(workers)} workers at {len(bases)} bases")
print(workers.groupby("partition").size().to_string())

## Preview
Grey dots are cabinets. Each pin is a node's base; hover for its partition and number of workers (grey pin: no workers).

In [ ]:
def preview_map(cabinets, nodes, workers):
    fmap = folium.Map(location=[nodes["lat"].mean(), nodes["long"].mean()], zoom_start=9, tiles="cartodbpositron")
    for c in cabinets.itertuples():
        folium.CircleMarker((c.cabinetlat, c.cabinetlong), radius=2, weight=0, fill=True, fill_opacity=0.5,
                            fill_color="#9aa0a6", tooltip=f"{c.CabinetId} ({c.node})").add_to(fmap)
    for p in nodes.itertuples():
        folium.Marker((p.lat, p.long), tooltip=f"{p.Index} ({p.partition}): {p.workers} workers, {p.demand:.1f} tickets/day",
                      icon=folium.Icon(color="darkblue" if p.workers else "gray", icon="home")).add_to(fmap)
    return fmap


preview_map(cabinets, nodes, workers)

## Save
The file goes to `OUTPUT_FILE`.

In [ ]:
out_path = ROOT / OUTPUT_FILE
out_path.parent.mkdir(parents=True, exist_ok=True)
workers.to_csv(out_path, index=False)
print(f"Saved {len(workers)} workers to {OUTPUT_FILE}")